# 02 · 万物皆 Token：四种模态的离散化之旅

**硬件**：🟢 CPU 可跑（VAE 权重约 300MB、EnCodec 约 100MB）

现代多模态模型的第一性原理：**把任何模态变成 token 序列，剩下的交给 Transformer**。本 notebook 亲手完成四种转换：

| 模态 | 方法 | 本节实践 |
|---|---|---|
| 文本 | BPE 子词 | GPT-2 tokenizer |
| 图像（理解侧） | ViT patch 切块 | 手写 patchify |
| 图像（生成侧） | VAE 潜空间 | SD VAE 编解码 |
| 音频（生成侧） | 神经 codec 离散 token | EnCodec |

对应理论：[theory.md](../theory.md) 第 1 节。

In [ ]:
%pip install -q torch transformers diffusers datasets soundfile matplotlib pillow requests

In [ ]:
import torch
import matplotlib.pyplot as plt

device = "cuda" if torch.cuda.is_available() else "cpu"  # 本节全程 CPU 也很快
print(f"device = {device}")

## 1. 文本 → BPE token

BPE 从字符开始，反复合并高频相邻对，得到子词表。观察两个现象：

1. 常见英文词是一个 token，生僻词被拆碎
2. GPT-2 的词表对中文极不友好——一个汉字常要 2–3 个 byte-level token（现代模型如 Qwen 的词表已大幅优化中文效率）

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("gpt2")

for text in [
    "Multimodal models turn everything into tokens.",
    "Antidisestablishmentarianism",
    "多模态模型把一切变成 token。",
]:
    ids = tok.encode(text)
    pieces = [tok.decode([i]) for i in ids]
    print(f"{text}\n  -> {len(ids)} tokens: {pieces}\n")

## 2. 图像 → ViT patch（理解侧）

ViT 的做法简单粗暴：把图像切成 16×16（或 14×14）的小方块，每块拉平后过一个线性层，就成了"视觉词"。一张 224×224 的图 = **196 个 patch token**——这个数字决定了 VLM 处理图像的成本。

In [ ]:
import requests
from io import BytesIO
from PIL import Image
import numpy as np

url = "http://images.cocodataset.org/val2017/000000039769.jpg"
img = Image.open(BytesIO(requests.get(url, timeout=30).content)).convert("RGB").resize((224, 224))
x = torch.tensor(np.array(img), dtype=torch.float32).permute(2, 0, 1) / 255.0  # [3,224,224]

P = 16
patches = x.unfold(1, P, P).unfold(2, P, P)          # [3, 14, 14, 16, 16]
patches = patches.permute(1, 2, 0, 3, 4)             # [14, 14, 3, 16, 16]
print(f"patch 网格: {patches.shape[0]}x{patches.shape[1]} = {patches.shape[0]*patches.shape[1]} 个 token")
print(f"每个 patch 拉平后维度: {3*P*P}（再过线性层投到模型维度）")

fig, axes = plt.subplots(14, 14, figsize=(6, 6))
for i in range(14):
    for j in range(14):
        axes[i, j].imshow(patches[i, j].permute(1, 2, 0))
        axes[i, j].axis("off")
plt.suptitle("一张图 = 196 个 patch token")
plt.show()

**思考**：高分辨率文档（如 1536×1536）按 16px 切就是 9216 个 token——这就是 01/02 章里"动态分辨率"和"视觉 token 压缩"技术存在的原因。

## 3. 图像 → VAE latent（生成侧）

生成模型不在像素上工作——太贵。Stable Diffusion 的 VAE 把 512×512×3 的图压成 64×64×4 的潜变量（**48 倍压缩**），扩散过程全部发生在这个潜空间里。

In [ ]:
from diffusers import AutoencoderKL

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse").to(device).eval()

xb = (x.unsqueeze(0).to(device) * 2 - 1)  # VAE 输入范围 [-1, 1]
with torch.no_grad():
    latent = vae.encode(xb).latent_dist.sample()
    recon = vae.decode(latent).sample

print(f"像素: {tuple(xb.shape)} = {xb.numel():,} 个值")
print(f"latent: {tuple(latent.shape)} = {latent.numel():,} 个值（压缩 {xb.numel()/latent.numel():.0f}x）")

fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(img); axes[0].set_title("原图"); axes[0].axis("off")
for c in range(4):
    axes[c+1].imshow(latent[0, c].cpu(), cmap="RdBu")
    axes[c+1].set_title(f"latent ch{c}"); axes[c+1].axis("off")
rec = ((recon[0].cpu().clamp(-1, 1) + 1) / 2).permute(1, 2, 0)
axes[5].imshow(rec); axes[5].set_title("VAE 重建"); axes[5].axis("off")
plt.show()

注意 latent 的 4 个通道隐约保留了空间结构——它不是玄学压缩，而是"带语义的缩略图"。生成侧的 DiT 就在这个 28×28×4（对 224 输入）的张量上做 flow matching，再 patchify 一次变成 DiT 的输入 token。

## 4. 音频 → EnCodec 离散 token

音频生成的基石：神经 codec 用**残差向量量化（RVQ）**把波形变成多层离散 token。第一层码本抓大结构，后面每层量化前面的残差——层数越多音质越好。这正是 06 章 TTS（VALL-E 范式）生成的目标序列。

In [ ]:
from transformers import EncodecModel, AutoProcessor
from datasets import load_dataset, Audio

codec = EncodecModel.from_pretrained("facebook/encodec_24khz").to(device).eval()
codec_proc = AutoProcessor.from_pretrained("facebook/encodec_24khz")

ds = load_dataset("hf-internal-testing/librispeech_asr_dummy", "clean", split="validation")
ds = ds.cast_column("audio", Audio(sampling_rate=24000))
wav = ds[0]["audio"]["array"]
print(f"音频时长: {len(wav)/24000:.1f}s")

inputs = codec_proc(raw_audio=wav, sampling_rate=24000, return_tensors="pt").to(device)
with torch.no_grad():
    enc = codec.encode(inputs["input_values"], inputs["padding_mask"], bandwidth=6.0)

codes = enc.audio_codes  # [chunks, batch, n_q, frames]
print(f"codes shape: {tuple(codes.shape)}")
print(f"-> {codes.shape[2]} 层 RVQ 码本 x 每秒 75 帧，码本大小 1024")
print(f"前 10 个 token（第一层）: {codes[0, 0, 0, :10].tolist()}")

In [ ]:
# 听觉对比：不同带宽（= 不同 RVQ 层数）的重建质量
from IPython.display import Audio as AudioPlayer, display

print("原始音频:")
display(AudioPlayer(wav, rate=24000))

for bw in [1.5, 6.0, 24.0]:
    with torch.no_grad():
        e = codec.encode(inputs["input_values"], inputs["padding_mask"], bandwidth=bw)
        d = codec.decode(e.audio_codes, e.audio_scales, inputs["padding_mask"])[0]
    n_q = e.audio_codes.shape[2]
    print(f"bandwidth={bw}kbps（{n_q} 层码本，每秒 {n_q*75} 个 token）:")
    display(AudioPlayer(d[0].cpu().numpy(), rate=24000))

## 总结：一张"token 汇率表"

| 内容 | token 数（量级） |
|---|---|
| 一句英文（10 词） | ~13 |
| 一张 224×224 图（ViT-16） | 196 |
| 一张 1536×1536 文档图 | ~9,000（压缩前） |
| 一秒音频（EnCodec 6kbps） | 600（8 层 × 75 帧） |
| 一秒 480p 视频（时空 patch） | ~10,000+ |

这张表解释了本教程后面几乎所有的工程决策：为什么 VLM 要压视觉 token（01 章）、为什么视频生成最烧钱（05 章）、为什么"把长文本渲染成图"反而能省 token（02 章的光学压缩）。

## 练习

1. 把 tokenizer 换成 `Qwen/Qwen3-0.6B` 的，对比中文效率。
2. 用 VAE latent 做算术：两张图的 latent 插值后解码，看看会得到什么。
3. EnCodec 用 1 层码本（bandwidth=0.75？试试最低档）重建，听听丢掉的是什么信息——音色还是内容？